# 05 — Evaluation

Goal: deep-dive into the best model's performance with **PR-AUC as the headline metric**, plus threshold tuning, feature importance, and SHAP-based interpretability.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import os
import shap
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_recall_curve,
    average_precision_score, roc_auc_score, f1_score
)

sns.set_theme(style='whitegrid')
%matplotlib inline

REPORT_DIR = '../reports/05_evaluation'
os.makedirs(REPORT_DIR, exist_ok=True)

## 1. Load Model, Metadata, and Test Data

The metadata from `04_modeling.ipynb` tells us which feature set (`full` or `selected`) the winning model was trained on, so we load the matching CSVs.

In [ ]:
with open('../models/best_model_metadata.json') as f:
    metadata = json.load(f)

feature_set = metadata['feature_set']
print(f"Best model: {metadata['model']} | strategy: {metadata['imbalance_strategy']} | feature_set: {feature_set}")
print(f"Features used ({len(metadata['features'])}): {metadata['features']}")

model = joblib.load('../models/best_model.pkl')

train = pd.read_csv(f'../data/processed/train_{feature_set}.csv')
test = pd.read_csv(f'../data/processed/test_{feature_set}.csv')

X_train = train.drop(columns=['Class'])
y_train = train['Class']
X_test = test.drop(columns=['Class'])
y_test = test['Class']

y_prob = model.predict_proba(X_test)[:, 1]

print(f'\nPR-AUC : {average_precision_score(y_test, y_prob):.4f}')
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')

## 2. Precision-Recall Curve (Headline Metric)

PR-AUC is the primary evaluation metric for this project — accuracy and ROC-AUC are both misleadingly optimistic under ~0.17% fraud prevalence.

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
prauc = average_precision_score(y_test, y_prob)

plt.figure(figsize=(8, 5))
plt.plot(recall, precision, label=f'PR-AUC = {prauc:.4f}', color='steelblue')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/pr_curve.png', dpi=150)
plt.show()

## 3. Threshold Tuning

Find the threshold that maximizes F1-score.

In [ ]:
f1_scores = [f1_score(y_test, (y_prob >= t).astype(int)) for t in thresholds]
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]

print(f'Best threshold: {best_threshold:.4f}')
print(f'Best F1:        {best_f1:.4f}')

plt.figure(figsize=(8, 4))
plt.plot(thresholds, f1_scores, color='tomato')
plt.axvline(best_threshold, linestyle='--', color='gray', label=f'Best threshold={best_threshold:.3f}')
plt.xlabel('Threshold')
plt.ylabel('F1 Score')
plt.title('F1 Score vs Threshold')
plt.legend()
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/threshold_tuning.png', dpi=150)
plt.show()

## 4. Confusion Matrix (Best Threshold)

Shown once, at the F1-optimal threshold, as a supplementary reference — PR-AUC above is the metric that drives model selection.

In [ ]:
y_pred_best = (y_prob >= best_threshold).astype(int)
cm_best = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(5, 4))
sns.heatmap(cm_best, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Legit', 'Fraud'],
            yticklabels=['Legit', 'Fraud'])
plt.title(f'Confusion Matrix (threshold={best_threshold:.3f})')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/confusion_matrix.png', dpi=150)
plt.show()

print(classification_report(y_test, y_pred_best, target_names=['Legit', 'Fraud']))

## 5. Feature Importance

In [ ]:
fitted_clf = model.named_steps['clf'] if hasattr(model, 'named_steps') else model

if hasattr(fitted_clf, 'feature_importances_'):
    importances = pd.Series(fitted_clf.feature_importances_, index=X_test.columns)
    importances = importances.sort_values(ascending=False).head(20)

    plt.figure(figsize=(8, 6))
    importances.plot(kind='barh', color='steelblue')
    plt.title('Top 20 Feature Importances')
    plt.xlabel('Importance')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(f'{REPORT_DIR}/feature_importance.png', dpi=150)
    plt.show()
else:
    # Logistic Regression: use absolute coefficient values
    coef = pd.Series(np.abs(fitted_clf.coef_[0]), index=X_test.columns)
    coef = coef.sort_values(ascending=False).head(20)

    plt.figure(figsize=(8, 6))
    coef.plot(kind='barh', color='steelblue')
    plt.title('Top 20 Feature Coefficients (|coef|)')
    plt.xlabel('|Coefficient|')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(f'{REPORT_DIR}/feature_importance.png', dpi=150)
    plt.show()

## 6. SHAP Analysis

Local and global interpretability for the winning model. The background sample (for the explainer baseline) and explained sample (rows actually attributed) are both subsampled for tractability.

In [ ]:
N_BACKGROUND = 300
N_EXPLAIN = 500

background = X_train.sample(n=min(N_BACKGROUND, len(X_train)), random_state=42)
explain_sample = X_test.sample(n=min(N_EXPLAIN, len(X_test)), random_state=42)

explainer = shap.Explainer(fitted_clf, background)
shap_values = explainer(explain_sample)

In [ ]:
plt.figure()
shap.summary_plot(shap_values, explain_sample, show=False)
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure()
shap.summary_plot(shap_values, explain_sample, plot_type='bar', show=False)
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/shap_importance_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
mean_abs_shap = pd.Series(np.abs(shap_values.values).mean(axis=0), index=explain_sample.columns)
top3_features = mean_abs_shap.sort_values(ascending=False).index[:3].tolist()
print(f'Top 3 features by mean |SHAP value|: {top3_features}')

for feat in top3_features:
    plt.figure()
    shap.dependence_plot(feat, shap_values.values, explain_sample, show=False)
    plt.tight_layout()
    plt.savefig(f'{REPORT_DIR}/shap_dependence_{feat}.png', dpi=150, bbox_inches='tight')
    plt.show()

### Local Explanations: a Caught Fraud vs. a Missed Fraud

In [ ]:
y_explain = y_test.loc[explain_sample.index]
prob_explain = fitted_clf.predict_proba(explain_sample)[:, 1]
pred_explain = (prob_explain >= best_threshold).astype(int)

true_positive_idx = explain_sample.index[(y_explain == 1) & (pred_explain == 1)]
false_negative_idx = explain_sample.index[(y_explain == 1) & (pred_explain == 0)]

if len(true_positive_idx) > 0:
    pos = explain_sample.index.get_loc(true_positive_idx[0])
    shap.plots.waterfall(shap_values[pos], show=False)
    plt.title('Correctly Identified Fraud (True Positive)')
    plt.tight_layout()
    plt.savefig(f'{REPORT_DIR}/shap_waterfall_true_positive.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No true positive fraud case found in the explained sample.')

if len(false_negative_idx) > 0:
    pos = explain_sample.index.get_loc(false_negative_idx[0])
    shap.plots.waterfall(shap_values[pos], show=False)
    plt.title('Missed Fraud (False Negative)')
    plt.tight_layout()
    plt.savefig(f'{REPORT_DIR}/shap_waterfall_false_negative.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No false negative fraud case found in the explained sample.')